In [1]:
import torch             #PyTorch Hauptlibrary
import torch.nn as nn    #nn = Neuronales Netz
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter #weiß nict ob wirs brauchen tbh

from collections import deque
import numpy as np       #Mathe
import random
import os

import gymnasium as gym #Trainingsumgebung

#Reproduzierbarkeit
seed = 1
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

#deterministic behavior for CUDA operations
#https://discuss.pytorch.org/t/what-is-the-differenc-between-cudnn-deterministic-and-cudnn-benchmark/38054/2
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [2]:
%load_ext tensorboard
%xmode Verbose

Exception reporting mode: Verbose


In [3]:
#GPU-Nutzung prüfen und einstellen
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

Using device: cuda


In [4]:
#Hyperparameter
env_name = "LunarLander-v3"
gamma = 0.99 #Discount Faktor
learning_rate = 0.00025 #Gewichtsanpassung

#Epsilon Greedy
eps_start = 0.70 #Start: Nur Ausprobieren
eps = eps_start
eps_end = 0.05

train_freq = 1
min_replay_size = 1000
batch_size = 64
gamma = 0.99
target_update_freq = 500 #Vorher 1000
batch_size = 64
buffer_size = 200_000
min_replay_size = 1_000
train_freq = 1

max_episodes = 1000 #
max_steps = 1000
train_episodes = max_episodes
#decay_factor = (eps_start - eps_end) / 12_000 #lin
decay_factor = (eps_end/eps_start)**(1/12_000) #exp

In [5]:
env = gym.make(env_name)
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

DependencyNotInstalled: Box2D is not installed, you can install it by run `pip install swig` followed by `pip install "gymnasium[box2d]"`

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QNetwork, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim) #output = die Anzahl aller möglichen Aktionen
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity): #capacity ist die Länge der dequeue
        self.buffer = deque(maxlen=capacity) #double ended queue

    def push(self, state, action, reward, next_state, done): #siehe http://www.incompleteideas.net/book/ebook/figtmp7.png
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        samples = random.sample(self.buffer, batch_size) #random weil siehe Doku Quellen dazu
        states, actions, rewards, next_states, dones = zip(*samples)
        return (
            torch.tensor(states, dtype=torch.float32, device=device),
            torch.tensor(actions, dtype=torch.int64, device=device).unsqueeze(1), #https://docs.pytorch.org/docs/main/generated/torch.unsqueeze.html
            torch.tensor(rewards, dtype=torch.float32, device=device).unsqueeze(1),
            torch.tensor(next_states, dtype=torch.float32, device=device),
            torch.tensor(dones, dtype=torch.float32, device=device).unsqueeze(1),
        )

    def __len__(self):
        return len(self.buffer)

In [ ]:
def getEpsilonLin():
    global eps
    eps = max(eps - decay_factor, eps_end)
    return eps

In [ ]:
# nicht linear
def getEpsilonExp():
    global eps
    eps = max(eps * decay_factor, eps_end)
    return eps

In [ ]:
policy_net = QNetwork(state_dim, action_dim).to(device)
target_net = QNetwork(state_dim, action_dim).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()
optimizer = optim.Adam(policy_net.parameters(), lr=learning_rate)
replay_buffer = ReplayBuffer(buffer_size)
writer = SummaryWriter()

global_step = 0
episode_rewards = []

In [ ]:
def soft_update(target_net, online_net, tau):
    for target_param, policy_param in zip(target_net.parameters(), policy_net.parameters()):
        policy_param.data.copy_(
            tau * policy_param.data + (1.0 - tau) * target_param.data
        )

In [ ]:
def train():
  global global_step
  print("\n Training...\n")
  q_values_episode = []
  for episode in range(max_episodes):
      state, _ = env.reset()
      state = np.array(state, dtype=np.float32)
      total_reward = 0

      for step in range(max_steps):
          epsilon = getEpsilonExp()   #exp pro Step
          global_step += 1

          #Action wählen
          if random.random() < epsilon:
              action = env.action_space.sample()
          else:
              with torch.no_grad():
                  state_tensor = torch.tensor(state, device=device).unsqueeze(0)
                  q_values = policy_net(state_tensor)
                  action = q_values.argmax(1).item()

          #Schritt im Environment
          next_state, reward, terminated, truncated, _ = env.step(action)
          done = terminated or truncated
          next_state = np.array(next_state, dtype=np.float32)

          #Reward Clipping derzeit aus, für cartpole nicht nötig
          #clipped_reward = max(min(reward, 1.0), -1.0)
          clipped_reward = reward

          #In Replay Buffer speichern
          replay_buffer.push(state, action, clipped_reward, next_state, done)

          state = next_state
          total_reward += reward

          #Trainingz
          if len(replay_buffer) >= min_replay_size and global_step % train_freq == 0:
              states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

              with torch.no_grad():
                  #Double DQN: Aktion über policy, Q über target
                  next_actions = policy_net(next_states).argmax(1, keepdim=True)
                  next_q_values = target_net(next_states).gather(1, next_actions)
                  target_q = rewards + (1 - dones) * gamma * next_q_values

              current_q = policy_net(states).gather(1, actions)
              loss = nn.SmoothL1Loss()(current_q, target_q)  #Huber

              optimizer.zero_grad()
              loss.backward()
              optimizer.step()

          #Target Network Update
          if global_step % target_update_freq == 0:
              target_net.load_state_dict(policy_net.state_dict())
          #soft_update(target_net, policy_net, tau=0.005)

          with torch.no_grad(): #xxxx
            q_values = policy_net(torch.tensor(state, device=device).unsqueeze(0))
            max_q_value = q_values.max(1)[0].item()

            q_values_episode.append(max_q_value)

          if done:
              break

      #Logging
      writer.add_scalar("Reward", total_reward, episode)
      writer.add_scalar("Epsilon", epsilon, episode)
      episode_rewards.append(total_reward)
      avg_q_value = np.mean(q_values_episode) #xx
      writer.add_scalar("Avg_Q_Value", avg_q_value, episode) #xx

      print(f"Episode {episode} - Total Reward: {total_reward:.2f} - Epsilon: {epsilon:.3f} - Avg_Q_Value: {avg_q_value}")

  env.close()
  writer.close()

In [ ]:
def evaluate(num_episodes=10):
    print("\n Evaluating...\n")
    total_eval_reward = 0

    for episode in range(num_episodes):
      state, _ = env.reset()
      state = np.array(state, dtype=np.float32)
      episode_reward = 0

      for step in range(max_steps):
          with torch.no_grad():
              state_tensor = torch.tensor(state, device=device).unsqueeze(0)
              q_values = policy_net(state_tensor)
              action = q_values.argmax(1).item()

          next_state, reward, terminated, truncated, _ = env.step(action)
          done = terminated or truncated
          state = np.array(next_state, dtype=np.float32)
          episode_reward += reward

          if done:
            break

      print(f"Evaluation Episode {episode +1} - Reward: {episode_reward:.2f}")
      total_eval_reward += episode_reward
    avg_eval = total_eval_reward / num_episodes
    print(f"Average Evaluation Reward: {avg_eval:.2f}")

In [ ]:
train()
evaluate(10)